# Calculatrice seq2seq

Dans ces travaux pratiques, un réseau récurrent apprend à calculer à partir de texte : il lit une équation écrite caractère par caractère, comme `12+34`, et écrit son résultat, `46`, caractère par caractère lui aussi. C'est un modèle *sequence to sequence* (seq2seq) : un encodeur résume la séquence d'entrée, un décodeur produit la séquence de sortie.

1. Encoder les équations.
2. Générer le jeu de données.
3. Construire l'encodeur-décodeur.
4. L'entraîner, puis l'évaluer.
5. La réussite par opération.
6. Faut-il inverser l'équation ?

Ce TP a besoin d'une carte graphique : vérifiez dans `Exécution > Modifier le type d'exécution` que l'accélérateur matériel est un GPU. Ses deux entraînements y prennent en tout environ deux minutes et demie sur une RTX 3090 ; le T4 de Colab est environ deux à trois fois plus lent.

## Import des bibliothèques

In [ ]:
import operator

import keras
import matplotlib.pyplot as plt
import numpy as np
import sklearn.model_selection
from keras import layers

# Mêmes tirages aléatoires à chaque exécution : des résultats reproductibles
keras.utils.set_random_seed(0)

## 1. Encoder les équations

Les équations et leurs résultats s'écrivent avec 16 caractères : les chiffres, le point décimal, les 4 opérations et l'espace, qui sert de *padding*. `encode` transforme une chaîne en tableau d'indices de ce vocabulaire, `decode` fait l'inverse.

In [ ]:
operations = list("+*-/")
numbers = list("0123456789.")
padding = [" "]

index_to_char = numbers + operations + padding
char_to_index = {char: index for index, char in enumerate(index_to_char)}


def encode(characters: str) -> np.ndarray:
  return np.array([char_to_index[char] for char in characters])


def decode(indices: np.ndarray) -> str:
  return "".join(index_to_char[index] for index in indices)


print(f"Vocabulaire : {index_to_char}")
print(f'encode("3/4+2-5") = {encode("3/4+2-5")}')
print(f'decode(encode("3/4+2-5")) = {decode(encode("3/4+2-5"))}')

### Quelle entrée pour le réseau ?

Le réseau peut-il travailler directement sur la sortie de `encode`, des entiers de 0 à 15, ou faut-il une transformation de plus ? Laquelle ?

Votre réponse ici

#### Solution

Ces indices désignent des catégories, pas des quantités : que `+` ait l'indice 11 et `9` l'indice 9 ne veut pas dire que `+` vaut plus que `9`. Donnés tels quels, ils feraient croire au réseau à un ordre et à des distances qui n'existent pas. Il faut donc, au choix :

- un encodage *one-hot* : un vecteur de 16 valeurs, toutes nulles sauf un 1 à l'indice du caractère ;
- une couche `Embedding`, qui apprend un vecteur par caractère, comme dans les slides. C'est ce que fera notre modèle.

## 2. Générer le jeu de données

`make_maths_problem` tire une opération au hasard et deux entiers de 0 à 999, jusqu'à obtenir le nombre d'équations demandé, toutes différentes :

- les divisions par 0 sont écartées ;
- les résultats sont tronqués à 6 caractères (`0.3333`) ;
- équations et résultats sont complétés par des espaces, pour faire tous 7 et 6 caractères ;
- l'équation est inversée par défaut (`invert=True`) : `12+34` devient `43+21`. La partie 6 dit pourquoi.

In [ ]:
def make_maths_problem(
  n_samples: int = 1000, n_digits: int = 3, invert: bool = True
) -> tuple[np.ndarray, np.ndarray]:
  equations = []
  results = []
  seen = set()

  math_operation = {
    "+": operator.add,
    "-": operator.sub,
    "*": operator.mul,
    "/": operator.truediv,
  }

  # Longueurs maximales de l'équation et du résultat
  max_equation_len = 2 * n_digits + 1
  max_result_len = 2 * n_digits

  while len(equations) < n_samples:
    operation = np.random.choice(operations)
    left, right = np.random.randint(10**n_digits, size=2)
    equation = f"{left}{operation}{right}"
    # Une équation déjà tirée, ou une division par 0 : on recommence
    if equation in seen or (operation == "/" and right == 0):
      continue
    seen.add(equation)

    # Un résultat comme 0.3333333… est tronqué
    result = str(math_operation[operation](left, right))[:max_result_len]
    # Des espaces pour que toutes les séquences aient la même longueur
    padded_equation = equation.ljust(max_equation_len)
    padded_result = result.ljust(max_result_len)
    if invert:
      padded_equation = padded_equation[::-1]

    equations.append(padded_equation)
    results.append(padded_result)

  X = np.stack([encode(equation) for equation in equations])
  y = np.stack([encode(result) for result in results])
  return X, y


X, y = make_maths_problem(10)
print(f"Formes de X et y : {X.shape}, {y.shape}")
for encoded_equation, encoded_result in zip(X, y):
  # L'équation est inversée : on la remet dans l'ordre pour l'afficher
  print(f"{decode(encoded_equation[::-1])} = {decode(encoded_result)}")

Le jeu de données : 100 000 équations, dont 80 % pour l'entraînement et 20 % pour le test.

In [ ]:
X, y = make_maths_problem(100_000)
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
  X, y, train_size=0.8, random_state=0
)
print(X_train.shape, X_test.shape)

## 3. Construire l'encodeur-décodeur

Écrivez la fonction `build_model(hidden_size=512)`, qui renvoie un modèle `keras.Sequential` compilé :

- **l'entrée** : des séquences de 7 indices (`max_equation_len`) ;
- **l'encodeur** : une couche `layers.Embedding` de 16 dimensions, puis deux couches `layers.GRU` de `hidden_size` unités empilées ; la seconde ne renvoie que son dernier état caché, le résumé de toute l'équation ;
- **le décodeur** : une couche `layers.RepeatVector` répète ce résumé 6 fois (`max_result_len`), une fois par caractère du résultat ; un GRU de `hidden_size` unités lit cette séquence et renvoie un état par pas ; une couche `layers.Dense`, appliquée à chaque pas, donne la probabilité de chacun des 16 caractères ;
- compilé avec la perte `sparse_categorical_crossentropy`, l'optimiseur `adam` et la justesse pour métrique.

Ce décodeur ne reçoit pas le caractère qu'il vient d'écrire : chaque pas reçoit le même résumé de l'équation, sans *teacher forcing*. C'est le plus simple des décodeurs : il suffit à l'addition et à la soustraction, pas à la multiplication ni à la division (partie 5).

**Documentation utile**

- [`keras.layers.RepeatVector`](https://keras.io/api/layers/reshaping_layers/repeat_vector/)

In [ ]:
max_equation_len, max_result_len = X.shape[1], y.shape[1]


def build_model(hidden_size: int = 512) -> keras.Sequential:
  # Votre code ici
  ...

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `keras.Input(shape=(max_equation_len,), dtype="int32")`
- `layers.Embedding(len(index_to_char), 16)`
- `layers.GRU(hidden_size, return_sequences=True)` renvoie un état par pas, la séquence que lit la couche suivante ; sans `return_sequences`, le dernier état seulement
- `layers.RepeatVector(max_result_len)` : de `(batch, hidden_size)` à `(batch, 6, hidden_size)`
- `layers.Dense(len(index_to_char), activation="softmax")` : appliquée à un tenseur `(batch, 6, hidden_size)`, une couche `Dense` agit sur la dernière dimension, donc sur chaque pas séparément ; la sortie est de forme `(batch, 6, 16)`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def build_model(hidden_size: int = 512) -> keras.Sequential:
  model = keras.Sequential(
    [
      keras.Input(shape=(...,), dtype="int32"),
      layers.Embedding(len(index_to_char), ...),
      layers.GRU(hidden_size, return_sequences=...),
      layers.GRU(hidden_size),
      layers.RepeatVector(...),
      layers.GRU(hidden_size, return_sequences=...),
      layers.Dense(..., activation=...),
    ]
  )
  model.compile(loss=..., optimizer="adam", metrics=["accuracy"])
  return model
```

### Solution

In [ ]:
def build_model(hidden_size: int = 512) -> keras.Sequential:
  model = keras.Sequential(
    [
      keras.Input(shape=(max_equation_len,), dtype="int32"),
      # Encodeur : son dernier état résume l'équation
      layers.Embedding(len(index_to_char), 16),
      layers.GRU(hidden_size, return_sequences=True),
      layers.GRU(hidden_size),
      # Décodeur : le résumé, répété pour chacun des caractères du résultat
      layers.RepeatVector(max_result_len),
      layers.GRU(hidden_size, return_sequences=True),
      # Appliquée à chaque pas : une distribution sur les 16 caractères
      layers.Dense(len(index_to_char), activation="softmax"),
    ]
  )
  model.compile(
    loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"]
  )
  return model

In [ ]:
model = build_model()
model.summary()

## 4. Entraîner, puis évaluer le modèle

Pendant l'entraînement, le callback `PrintExamples` affiche toutes les 10 epochs les résultats que le modèle donne pour les 20 mêmes équations de test : on le voit progresser. Les 50 epochs prennent environ une minute et demie sur une RTX 3090, trois à quatre minutes sur le T4 de Colab.

La justesse que Keras affiche est celle des caractères : un résultat dont un seul caractère est faux y compte pour 5/6.

In [ ]:
X_examples, y_examples = X_test[:20], y_test[:20]


def examples(model: keras.Model) -> None:
  guesses = model.predict(X_examples, verbose=0).argmax(axis=-1)
  for equation, result, guess in zip(X_examples, y_examples, guesses):
    print(f"{decode(equation[::-1])} = {decode(result)} ?= {decode(guess)}")


class PrintExamples(keras.callbacks.Callback):
  def __init__(self, every: int = 10) -> None:
    super().__init__()
    self.every = every

  def on_epoch_end(self, epoch: int, logs: dict | None = None) -> None:
    if (epoch + 1) % self.every == 0:
      print(f"\nAprès {epoch + 1} epochs :")
      examples(self.model)  # le modèle que fit entraîne


EPOCHS = 50
history = model.fit(
  X_train,
  y_train,
  batch_size=512,
  epochs=EPOCHS,
  validation_data=(X_test, y_test),
  callbacks=[PrintExamples(every=10)],
)

La fonction `evaluate` mesure, sur les équations de test :

- la proportion de prédictions illégales, qui ne se lisent pas comme un nombre (`1.2.3`) ;
- la justesse des résultats entiers : la proportion de résultats exacts, au caractère près ;
- l'erreur relative $\frac{y_{pred} - y}{y}$ des prédictions légales, dont elle trace la distribution.

In [ ]:
def evaluate(model: keras.Model) -> None:
  predictions = model.predict(X_test, batch_size=1024, verbose=0).argmax(axis=-1)
  exact = (predictions == y_test).all(axis=1)

  # Les prédictions qui se lisent comme un nombre, et leurs vrais résultats
  y_pred, y_true = [], []
  for prediction, result in zip(predictions, y_test):
    try:
      y_pred.append(float(decode(prediction)))
    except ValueError:
      continue
    y_true.append(float(decode(result)))
  y_pred, y_true = np.array(y_pred), np.array(y_true)
  illegal = 1 - len(y_pred) / len(y_test)
  print(f"Prédictions illégales : {illegal:.1%}")
  print(f"Résultats exacts : {exact.mean():.1%}")

  # Erreur relative, divisée par 1 quand le vrai résultat est nul
  relative_error = (y_pred - y_true) / np.where(y_true == 0, 1, y_true)
  print(f"Erreur relative absolue médiane : {np.median(np.abs(relative_error)):.2%}")

  fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
  # Des classes régulières sur l'axe logarithmique ; un résultat exact compte à 1e-4
  errors = np.maximum(np.abs(relative_error), 1e-4)
  ax1.hist(errors, bins=np.geomspace(1e-4, errors.max(), 50))
  ax1.set_xscale("log")
  ax1.set_yscale("log")
  ax1.set_title("Toutes les erreurs relatives, en valeur absolue")
  ax2.hist(relative_error, bins=20, range=(-0.1, 0.1))
  ax2.set_title("Zoom autour de 0")
  for ax in (ax1, ax2):
    ax.set_xlabel("Erreur relative")
    ax.set_ylabel("Équations")
  plt.show()


evaluate(model)

## 5. La réussite par opération

Le modèle réussit-il aussi bien les 4 opérations ? Calculez, pour chacune, la proportion d'équations de test dont le résultat prédit est **entièrement** juste, et expliquez les écarts.

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `model.predict(X_test).argmax(axis=-1)` : le caractère le plus probable à chaque pas, de forme `(20000, 6)`
- `(predictions == y_test).all(axis=1)` : vrai pour un résultat dont les 6 caractères sont justes
- `decode(equation[::-1])` remet une équation dans l'ordre, et `"+" in equation` dit si c'est une addition
- un tableau booléen sélectionne des lignes : `exact[mask]`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
predictions = model.predict(X_test, batch_size=1024, verbose=0).argmax(axis=-1)
exact = (predictions == ...).all(axis=...)
equations = [decode(equation[::-1]) for equation in X_test]
for operation in operations:
  mask = np.array([operation in equation for equation in ...])
  print(f"{operation} : {exact[...].mean():.1%}")
```

### Solution

In [ ]:
predictions = model.predict(X_test, batch_size=1024, verbose=0).argmax(axis=-1)
# Un résultat n'est juste que si ses 6 caractères le sont
exact = (predictions == y_test).all(axis=1)
equations = [decode(equation[::-1]) for equation in X_test]
for operation in operations:
  mask = np.array([operation in equation for equation in equations])
  print(f"{operation} : {exact[mask].mean():.1%} de résultats exacts")

Les écarts sont énormes : l'addition et la soustraction sont le plus souvent justes (80 à 90 % de résultats exacts), la multiplication et la division presque jamais (moins de 1 %). Dans une addition ou une soustraction, chaque chiffre du résultat ne dépend que de quelques chiffres des opérandes et d'une retenue. Dans une multiplication, chaque chiffre du résultat dépend de tous ceux des opérandes, et une division donne souvent des décimales sans structure simple : ces deux opérations sont bien plus difficiles.

Le modèle n'y est pourtant pas perdu : il trouve souvent l'ordre de grandeur et les premiers chiffres (`721/866 = 0.8325`, prédit `0.82…`), rarement tous. Le zoom de la partie 4 le montre : en dehors des résultats exacts, la plupart des erreurs relatives restent sous 10 %.

## 6. Faut-il inverser l'équation ?

`make_maths_problem` inverse les équations par défaut : l'encodeur lit `321+654` sous la forme `456+123`. Générez un jeu de données sans inversion, entraînez le même modèle dessus pendant 30 epochs, et comparez sa courbe de justesse de validation à celle du premier modèle. Comment expliquer l'écart ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `make_maths_problem(100_000, invert=False)` : les équations dans l'ordre
- `history.history["val_accuracy"]` : la justesse de validation de chaque epoch du premier modèle ; `[:30]` garde ses 30 premières

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
X_raw, y_raw = make_maths_problem(100_000, invert=...)
X_raw_train, X_raw_test, y_raw_train, y_raw_test = (
  sklearn.model_selection.train_test_split(X_raw, y_raw, train_size=0.8, random_state=0)
)
raw_history = build_model().fit(
  ..., ..., batch_size=512, epochs=30, validation_data=(..., ...)
)

plt.plot(history.history["val_accuracy"][:30], label=...)
plt.plot(raw_history.history[...], label=...)
plt.legend()
plt.show()
```

### Solution

In [ ]:
X_raw, y_raw = make_maths_problem(100_000, invert=False)
X_raw_train, X_raw_test, y_raw_train, y_raw_test = (
  sklearn.model_selection.train_test_split(X_raw, y_raw, train_size=0.8, random_state=0)
)
raw_history = build_model().fit(
  X_raw_train,
  y_raw_train,
  batch_size=512,
  epochs=30,
  validation_data=(X_raw_test, y_raw_test),
)

plt.plot(history.history["val_accuracy"][:30], label="Équation inversée")
plt.plot(raw_history.history["val_accuracy"], label="Équation dans l'ordre")
plt.xlabel("Epoch (à partir de 0)")
plt.ylabel("Justesse de validation (par caractère)")
plt.legend()
plt.show()

Le modèle apprend plus vite sur les équations inversées : après 10 epochs, environ 65 % des caractères de validation sont justes, contre 54 à 55 % dans l'ordre ; après 20, 73 à 74 % contre 65 %. Sur les équations dans l'ordre, il rattrape ensuite une partie de son retard (71 à 73 % à la 30e epoch).

Les premiers caractères du résultat dépendent surtout des premiers chiffres des opérandes, en particulier de ceux de l'opérande de gauche. Inversée, l'équation commence par l'opérande de droite et finit par celui de gauche : l'encodeur lit ces premiers chiffres en dernier, ils sont plus présents dans son dernier état caché, qui est tout ce que reçoit le décodeur. C'est l'astuce de l'article [*Sequence to Sequence Learning with Neural Networks*](https://arxiv.org/abs/1409.3215), qui inversait les phrases sources en traduction automatique.